# EXP-001 hybrid 검색 판정 근거

목적: dev-full 판정(SPEC 판정 절)과 multi-doc 관찰(판정에 쓰지 않음)을 기록한다. 설정과 판정 기준은 `docs/experiments/EXP-001.md`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from rag.bm25 import BM25Retriever
from rag.eval import metrics as m1
from rag.eval_multi import metrics as mm
from rag.index import load_cfg
from rag.search import query_list, rank_docs, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / "data/runs"
BASE = "kure-v1-fixed-512-64"
HYBRID = "kure-v1-fixed-512-64-hybrid"
COMPARE = RUNS / "exp001-compare.json"  # rag.eval.compare --out
COMPARE_MULTI = RUNS / "exp001-multidoc-compare.json"  # rag.eval_multi.compare --out
INDEX_DIR = ROOT / "data/index/kure-v1-fixed-512-64/full"

## single-doc 판정 (dev-full)

`rag.eval.compare` 출력이다. 회의 단위 paired bootstrap 10,000회, seed 20260929.

In [3]:
cmp = json.loads(COMPARE.read_text(encoding="utf-8"))
pd.DataFrame({k: cmp[k]["metrics"] for k in ("base", "cand")}).assign(diff=lambda d: d.cand - d.base).round(4)

,base,cand,diff
recall@1,0.7062,0.7964,0.0902
recall@5,0.8584,0.9168,0.0584
recall@10,0.8909,0.9450,0.0540
mrr@10,0.7707,0.8495,0.0788


In [4]:
{"diff": round(cmp["diff"], 4), "ci": [round(x, 4) for x in cmp["ci"]], "verdict": cmp["verdict"]}

{'diff': 0.0584, 'ci': [0.0493, 0.0676], 'verdict': '채택'}

### 층별 Recall@5

In [5]:
cfg = load_cfg()
ev = cfg.eval
gold = m1.load_gold(cfg, "dev-full")
n_min = m1.min_docs(ev.recall_ks, ev.mrr_k)
ranks = {name: m1.gold_ranks(m1.read_run(RUNS / f"{run}-dev-full.csv"), gold, n_min)
         for name, run in (("dense", BASE), ("hybrid", HYBRID))}
scores = {name: m1.per_query(r, ev.recall_ks, ev.mrr_k) for name, r in ranks.items()}
tables = {name: m1.metric_table(s, gold, ev.breakdowns).set_index(["group", "value"])
          for name, s in scores.items()}
r5 = pd.DataFrame({"n": tables["dense"]["n"], "dense": tables["dense"]["recall@5"],
                   "hybrid": tables["hybrid"]["recall@5"]})
r5.assign(diff=r5.hybrid - r5.dense).round(4)

n   dense  hybrid    diff
group        value                                  
all          all        3016  0.8584  0.9168  0.0584
meeting_name 국정감사       1225  0.8824  0.9257  0.0433
             본회의         112  0.7768  0.8839  0.1071
             소위원회       1214  0.8674  0.9267  0.0593
             예산결산특별위원회   158  0.8038  0.8797  0.0759
             특별위원회       307  0.7850  0.8730  0.0879
qna_type     단답형         289  0.8408  0.8754  0.0346
             추출형        2727  0.8603  0.9212  0.0609

### 상위 5위 진입·이탈

hybrid가 dense의 5위 안 정답을 밀어낸 질의(이탈)와 5위 밖 정답을 끌어올린 질의(진입) 수.

In [6]:
hit = pd.DataFrame({name: s["recall@5"] for name, s in scores.items()})
pd.crosstab(hit["dense"], hit["hybrid"], rownames=["dense@5"], colnames=["hybrid@5"])

hybrid@5,0.0,1.0
dense@5,,
0.0,229,198
1.0,22,2567


## multi-doc 관찰 (판정에 쓰지 않음)

`rag.eval_multi.compare` 출력이다. 재표본 단위 `pool_id`(D-16).

In [7]:
cm = json.loads(COMPARE_MULTI.read_text(encoding="utf-8"))
d = pd.DataFrame(cm["diffs"])
d["ci"] = d["ci"].map(lambda c: f"[{c[0] * 100:+.2f}, {c[1] * 100:+.2f}]")
d["diff"] = (d["diff"] * 100).round(2)
d[d.metric.isin(["recall@5", "hit@5", "complete@5"])].pivot(index=["value", "n"], columns="metric", values=["diff", "ci"])

diff                              ci                   \
metric         complete@5  hit@5 recall@5      complete@5            hit@5   
value      n                                                                 
all        600      -1.83  -4.33    -3.21  [-4.17, +0.50]   [-7.17, -1.50]   
conf       233      -3.43  -1.29    -2.29  [-8.15, +1.29]   [-3.43, +0.86]   
law        152      -1.97 -13.82    -7.59  [-5.92, +1.97]  [-21.05, -6.58]   
questioner 215        0.0  -0.93     -1.1  [-3.26, +3.72]   [-6.05, +4.19]   

                                 
metric                 recall@5  
value      n                     
all        600   [-5.03, -1.42]  
conf       233   [-4.94, +0.36]  
law        152  [-11.69, -3.52]  
questioner 215   [-4.12, +1.95]

### 검색기별 multi-doc Recall@5

dense 단독(기준), BM25 단독, hybrid를 유형별로 나란히 본다. BM25 단독 순위는 같은 BM25 인덱스로 여기서 계산한다(약 1분).

In [8]:
mcfg = load_cfg(["+exp=exp001", "index.scope=full", "search.queries=multidoc"])
mgold = mm.load_gold(ROOT / mcfg.paths.multidoc_frozen)
chunk_doc_ids = np.array([json.loads(line)["doc_id"] for line in (INDEX_DIR / "chunks.jsonl").open(encoding="utf-8")])
bm25 = BM25Retriever(mcfg, INDEX_DIR)
qids, text_path = query_list(mcfg)
q_ids = bm25.query_ids(read_texts(text_path, qids))
bm25_docs = pd.Series([rank_docs(bm25.scores(i), chunk_doc_ids, 10, mcfg.retriever.chunk_pool) for i in q_ids], index=qids)
len(bm25_docs), bm25_docs.map(len).min()

(600, np.int64(10))

In [9]:
ks = mcfg.multidoc.eval.ks
multi = {
    "dense": mm.per_query(mm.ranked_docs(mm.read_run(RUNS / f"{BASE}-multidoc.csv"), mgold, max(ks)), mgold, ks),
    "bm25": mm.per_query(bm25_docs.reindex(mgold.index), mgold, ks),
    "hybrid": mm.per_query(mm.ranked_docs(mm.read_run(RUNS / f"{HYBRID}-multidoc.csv"), mgold, max(ks)), mgold, ks),
}
pd.DataFrame({name: s["recall@5"].groupby(mgold["type"]).mean() for name, s in multi.items()}).assign(n=mgold["type"].value_counts()).round(4)

,dense,bm25,hybrid,n
type,,,,
conf,0.8288,0.6612,0.8059,233
law,0.2990,0.1208,0.2231,152
questioner,0.1900,0.1036,0.1790,215


## BM25 점수가 0보다 큰 문서 수

검색기마다 상위 100개 문서를 결합에 넣는다. 질의 어휘와 겹치는 문서가 100개보다 적으면 점수 0인 문서가 BM25 순위를 채워 RRF 점수를 받는다(PR 리뷰 가이드 지적). 해당 질의 수를 센다.

In [10]:
def n_matched(cfg):
    qids, text_path = query_list(cfg)
    ids = bm25.query_ids(read_texts(text_path, qids))
    return pd.Series([len(set(chunk_doc_ids[bm25.scores(i) > 0])) for i in ids], index=qids)

for name, c in (("dev-full", load_cfg(["+exp=exp001", "index.scope=full"])), ("multidoc", mcfg)):
    n = n_matched(c)
    print(name, len(n), "100개 미만", (n < 100).sum(), "최소", n.min())

dev-full 3016 100개 미만 0 최소 897
multidoc 600 100개 미만 0 최소 33158
